# 04 — Streaming source and data-product research

A read-only research lab over one Kafka trade source per run. It connects source quality to market behavior, then turns measured evidence into candidate Bronze, Silver, Gold, and observability products.

The analysis has six parts: **Source and coverage**, **Freshness and latency**, **Uniqueness and integrity**, **Market activity**, **Cross-venue market structure**, and **Data-product evolution**. Cross-venue differences are research observations, **not executable arbitrage**; fees, depth, transfer constraints, and order latency are outside this dataset.

## 1. Configure one target

Choose `local` for the Docker broker or `msk` for the deployed AWS cluster. `quick` favors iteration; `deep` collects a longer research window. Every Kafka read remains bounded by both records and wall time.

In [ ]:
from __future__ import annotations

import warnings
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import devlab
from devlab import frames, health
from devlab.frames import NATURAL_KEY

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
TARGET = "local"
RUN_MODE = "quick"
TOPIC = "md.trades.v1"
PROFILES = {
    "quick": {"limit": 20_000, "seconds": 60.0},
    "deep": {"limit": 200_000, "seconds": 600.0},
}

if TARGET not in {"local", "msk"}:
    raise ValueError("TARGET must be 'local' or 'msk'")
if RUN_MODE not in PROFILES:
    raise ValueError("RUN_MODE must be 'quick' or 'deep'")

PROFILE = PROFILES[RUN_MODE]
target = devlab.local() if TARGET == "local" else devlab.from_terraform()
target

## 2. Preflight: broker, topic, partitions, and live arrivals

Retained records answer whether the log contains data. The live-rate sample reads from `latest`, answering whether new data is arriving now. Partition watermarks expose key-distribution skew before analysis begins.

In [ ]:
topic_inventory = frames.frame(devlab.topics(target))
selected_topic = topic_inventory[topic_inventory["name"] == TOPIC]
if selected_topic.empty:
    raise RuntimeError(f"Topic {TOPIC!r} was not found on target {TARGET!r}.")

partition_watermarks = frames.frame(devlab.partitions(target, TOPIC))
live_rate = devlab.rate(target, TOPIC, seconds=10.0)
print(
    f"target={TARGET} topic={TOPIC} retained={int(selected_topic.iloc[0]['messages']):,} "
    f"live_rate={live_rate.per_second:,.1f}/s"
)
display(topic_inventory)
display(partition_watermarks)
display(
    frames.frame([{"venue": venue, "trades": count} for venue, count in live_rate.by_venue.items()])
)

## 3. Capture one bounded research window

The capture reads retained records from `earliest`; every downstream result uses this same window. `raw_df` retains duplicates for quality analysis, while `clean_df` removes natural-key duplicates for economic calculations.

In [ ]:
capture_started_at = datetime.now(UTC)
records = devlab.collect(
    target,
    TOPIC,
    limit=PROFILE["limit"],
    seconds=PROFILE["seconds"],
    offset_reset="earliest",
)
capture_finished_at = datetime.now(UTC)
raw_df = frames.trades_frame(records)
if raw_df.empty:
    raise RuntimeError(
        "No trades were captured; verify the preflight and producer before continuing."
    )
clean_df = frames.dedupe(raw_df)

event_duration = raw_df["event_ts"].max() - raw_df["event_ts"].min()
capture_summary = pd.DataFrame(
    [
        {
            "target": TARGET,
            "mode": RUN_MODE,
            "record_limit": PROFILE["limit"],
            "time_limit_s": PROFILE["seconds"],
            "captured_rows": len(raw_df),
            "deduplicated_rows": len(clean_df),
            "event_start": raw_df["event_ts"].min(),
            "event_end": raw_df["event_ts"].max(),
            "event_duration_s": event_duration.total_seconds(),
            "capture_wall_time_s": (capture_finished_at - capture_started_at).total_seconds(),
            "venues": raw_df["venue"].nunique(),
            "instruments": raw_df["instrument_id"].nunique(),
        }
    ]
)
capture_summary

## 4. Source and coverage

Profile who supplied the window, which instruments are represented, and how much economic activity each segment contains.

In [ ]:
coverage = (
    clean_df.groupby(["venue", "instrument_id"], observed=True)
    .agg(
        trades=("trade_id", "count"),
        base_volume=("size", "sum"),
        notional=("notional", "sum"),
        event_start=("event_ts", "min"),
        event_end=("event_ts", "max"),
    )
    .reset_index()
    .sort_values("notional", ascending=False)
)
null_profile = pd.DataFrame(
    {
        "missing": raw_df.isna().sum(),
        "missing_rate": raw_df.isna().mean(),
        "dtype": raw_df.dtypes.astype(str),
    }
).sort_values(["missing_rate", "missing"], ascending=False)
source_profile = (
    raw_df.groupby(["source", "is_backfill"], observed=True).size().rename("trades").reset_index()
)
source_profile["share"] = source_profile["trades"] / len(raw_df)

if len(raw_df) < 2_000 or event_duration.total_seconds() < 60:
    warnings.warn(
        "This is a small or short capture. Treat rankings and market conclusions as directional.",
        stacklevel=1,
    )

display(coverage)
display(source_profile)
display(null_profile.head(15))

Base volume is only comparable within the same instrument. Notional is more useful for ranking activity here, but it still assumes the configured instruments share a compatible quote-currency interpretation. Backfill and repair records are legitimate input and must retain their provenance in Bronze.

## 5. Freshness and latency

Separate typical latency from its tail and remember that this is exchange-to-ingest delay, not Kafka-only latency.

In [ ]:
def latency_quantiles(values: pd.Series) -> pd.Series:
    clean = values.dropna()
    return pd.Series(
        {
            "observations": len(clean),
            "median_ms": clean.quantile(0.50),
            "p90_ms": clean.quantile(0.90),
            "p95_ms": clean.quantile(0.95),
            "p99_ms": clean.quantile(0.99),
            "max_ms": clean.max(),
        }
    )


latency_rows = [
    {"scope": "all", "name": "all", **latency_quantiles(raw_df["latency_ms"]).to_dict()}
]
for venue, group in raw_df.groupby("venue", observed=True):
    latency_rows.append(
        {"scope": "venue", "name": str(venue), **latency_quantiles(group["latency_ms"]).to_dict()}
    )
for instrument, group in raw_df.groupby("instrument_id", observed=True):
    if len(group) >= 100:
        latency_rows.append(
            {
                "scope": "instrument",
                "name": str(instrument),
                **latency_quantiles(group["latency_ms"]).to_dict(),
            }
        )
latency_summary = pd.DataFrame(latency_rows)
latency_p99 = raw_df["latency_ms"].quantile(0.99)
negative_latency_count = int(raw_df["latency_ms"].lt(0).sum())
extreme_latency_count = int(raw_df["latency_ms"].gt(latency_p99).sum())
freshness_s = max(
    0.0,
    (pd.Timestamp.now(tz="UTC") - raw_df["event_ts"].max()).total_seconds(),
)
display(latency_summary)
print(
    f"negative latency rows={negative_latency_count:,}; "
    f"above-p99 rows={extreme_latency_count:,}; end freshness={freshness_s:,.1f}s"
)

plot_latency = raw_df["latency_ms"].clip(upper=latency_p99)
axis = plot_latency.hist(bins=60, figsize=(11, 3))
axis.set(title="Ingest latency (display clipped at p99)", xlabel="milliseconds", ylabel="trades")

The histogram is clipped only for display; `latency_summary` uses the full, unclipped data. Negative latency usually indicates clock disagreement. Tail latency combines exchange publication, network, connector, and timestamping delay—it is not Kafka broker latency alone.

## 6. Uniqueness and integrity

Measure duplicates, conflicts, valid sequence gaps, partition skew, and ordering behavior before trusting market aggregates.

In [ ]:
duplicate_mask = raw_df.duplicated(subset=NATURAL_KEY, keep=False)
duplicate_rows = raw_df.loc[duplicate_mask].copy()
conflict_fields = ["price_str", "size_str", "side", "event_ts"]
if duplicate_rows.empty:
    conflicting_keys = 0
else:
    duplicate_variants = duplicate_rows.groupby(NATURAL_KEY, observed=True)[
        conflict_fields
    ].nunique(dropna=False)
    conflicting_keys = int(duplicate_variants.gt(1).any(axis=1).sum())

gap_report = health.sequence_gaps(records)
partition_profile = (
    clean_df.groupby("kafka_partition", observed=True)
    .agg(
        records=("trade_id", "count"),
        notional=("notional", "sum"),
        first_offset=("kafka_offset", "min"),
        last_offset=("kafka_offset", "max"),
    )
    .reset_index()
)
partition_mean = partition_profile["records"].mean()
partition_cv = (
    float(partition_profile["records"].std(ddof=0) / partition_mean) if partition_mean else np.nan
)

arrival_df = pd.DataFrame(records)
event_time_regressions = int(
    arrival_df.groupby(["venue", "venue_symbol"], sort=False, observed=True)["event_ts_us"]
    .diff()
    .lt(0)
    .sum()
)
offset_regressions = int(
    arrival_df.groupby("kafka_partition", sort=False, observed=True)["kafka_offset"]
    .diff()
    .lt(0)
    .sum()
)
duplicate_rate = float(duplicate_mask.sum() / len(raw_df))
quality_metrics = pd.DataFrame(
    [
        {"metric": "duplicate_row_rate", "value": duplicate_rate, "observations": len(raw_df)},
        {
            "metric": "conflicting_duplicate_keys",
            "value": conflicting_keys,
            "observations": int(duplicate_mask.sum()),
        },
        {
            "metric": "sequence_missing",
            "value": gap_report.missing,
            "observations": gap_report.checked,
        },
        {
            "metric": "partition_count_cv",
            "value": partition_cv,
            "observations": len(partition_profile),
        },
        {
            "metric": "event_time_regressions",
            "value": event_time_regressions,
            "observations": len(arrival_df),
        },
        {
            "metric": "offset_regressions",
            "value": offset_regressions,
            "observations": len(arrival_df),
        },
    ]
)

assert (clean_df["size"] >= 0).all()
assert (clean_df["notional"] >= 0).all()
assert not clean_df.duplicated(subset=NATURAL_KEY).any()
display(quality_metrics)
display(partition_profile)
print(f"sequence venues skipped by design: {gap_report.skipped_venues or '(none)'}")

Duplicate rows are expected when REST repair or replay overlaps the live stream; conflicting values for the same natural key are more serious and belong in quarantine. Coinbase is deliberately skipped by replay gap detection: its sequence is connection-wide, while Kafka preserves order only within each keyed partition. Event-time regressions can be legitimate late arrivals; Kafka offset regressions within a partition should be zero.

## 7. Market activity

Study intensity, trade sizes, directional imbalance, returns, and realized volatility on the deduplicated event-time view.

In [ ]:
BUCKET_FREQ = "1min"
ROLLING_VOL_BUCKETS = 5
activity = (
    clean_df.set_index("event_ts")
    .groupby([pd.Grouper(freq=BUCKET_FREQ), "instrument_id"], observed=True)
    .agg(trades=("trade_id", "count"), base_volume=("size", "sum"), notional=("notional", "sum"))
    .reset_index()
)
activity_ranking = (
    clean_df.groupby("instrument_id", observed=True)
    .agg(trades=("trade_id", "count"), notional=("notional", "sum"))
    .sort_values("notional", ascending=False)
    .reset_index()
)
trade_distributions = (
    clean_df[["size", "notional"]]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    .rename_axis("quantile")
    .reset_index()
)
side_activity = (
    clean_df.assign(side_label=clean_df["side"].astype("string").str.lower())
    .groupby("side_label", observed=True)
    .agg(trades=("trade_id", "count"), notional=("notional", "sum"))
)
buy_notional = float(side_activity["notional"].get("buy", 0.0))
sell_notional = float(side_activity["notional"].get("sell", 0.0))
notional_imbalance = (
    (buy_notional - sell_notional) / (buy_notional + sell_notional)
    if buy_notional + sell_notional
    else np.nan
)

price_buckets = (
    clean_df.set_index("event_ts")
    .groupby("instrument_id", observed=True)["price"]
    .resample(BUCKET_FREQ)
    .last()
    .dropna()
    .rename("close")
    .reset_index()
)
price_buckets["return"] = price_buckets.groupby("instrument_id", observed=True)["close"].pct_change(
    fill_method=None
)
price_buckets["rolling_vol_bps"] = price_buckets.groupby("instrument_id", observed=True)[
    "return"
].transform(lambda values: values.rolling(ROLLING_VOL_BUCKETS, min_periods=3).std() * 10_000)
volatility_ranking = (
    price_buckets.groupby("instrument_id", observed=True)
    .agg(
        buckets=("close", "count"),
        return_observations=("return", "count"),
        median_rolling_vol_bps=("rolling_vol_bps", "median"),
        max_rolling_vol_bps=("rolling_vol_bps", "max"),
    )
    .query("return_observations >= 5")
    .sort_values("median_rolling_vol_bps", ascending=False)
    .reset_index()
)

display(activity_ranking)
display(trade_distributions)
display(side_activity)
display(volatility_ranking)
print(f"buy/sell notional imbalance={notional_imbalance:+.3f}")
if not activity.empty:
    activity_plot = activity.pivot(index="event_ts", columns="instrument_id", values="notional")
    activity_plot.plot(figsize=(12, 4), title="Notional activity per minute", alpha=0.8)
    plt.ylabel("notional")
    plt.show()

Volatility is the rolling standard deviation of one-minute event-time returns, expressed in basis points and intentionally not annualized. Instruments need at least five return observations to be ranked; an absent instrument is insufficient evidence, not zero volatility. Imbalance describes observed aggressive-side notional in this window and is not a prediction.

## 8. Cross-venue market structure

Compare synchronized venue VWAPs, spreads, persistence, and lagged returns without interpreting them as executable trades.

In [ ]:
venue_comparison = frames.venue_comparison(clean_df, freq=BUCKET_FREQ)
spread_rows = []
if "spread_bps" in venue_comparison.columns:
    synchronized_spreads = venue_comparison.dropna(subset=["spread_bps"]).copy()
    for instrument, group in synchronized_spreads.groupby("instrument_id", observed=True):
        signs = np.sign(group["spread_bps"])
        persistence = float(signs.eq(signs.shift()).iloc[1:].mean()) if len(signs) > 1 else np.nan
        spread_rows.append(
            {
                "instrument_id": str(instrument),
                "samples": len(group),
                "median_bps": group["spread_bps"].median(),
                "p90_abs_bps": group["spread_bps"].abs().quantile(0.90),
                "p95_abs_bps": group["spread_bps"].abs().quantile(0.95),
                "min_bps": group["spread_bps"].min(),
                "max_bps": group["spread_bps"].max(),
                "sign_persistence": persistence,
            }
        )
else:
    synchronized_spreads = pd.DataFrame()
spread_summary = pd.DataFrame(
    spread_rows,
    columns=[
        "instrument_id",
        "samples",
        "median_bps",
        "p90_abs_bps",
        "p95_abs_bps",
        "min_bps",
        "max_bps",
        "sign_persistence",
    ],
)

venue_prices = (
    clean_df.set_index("event_ts")
    .groupby([pd.Grouper(freq=BUCKET_FREQ), "instrument_id", "venue"], observed=True)["price"]
    .last()
    .dropna()
    .rename("close")
    .reset_index()
)
leadership_rows = []
for instrument, group in venue_prices.groupby("instrument_id", observed=True):
    wide_prices = group.pivot(index="event_ts", columns="venue", values="close").dropna()
    if len(wide_prices.columns) != 2 or len(wide_prices) < 11:
        continue
    venue_a, venue_b = [str(column) for column in wide_prices.columns]
    venue_returns = wide_prices.pct_change(fill_method=None).dropna()
    for lag in range(-2, 3):
        correlation = venue_returns.iloc[:, 0].corr(venue_returns.iloc[:, 1].shift(lag))
        leadership_rows.append(
            {
                "instrument_id": str(instrument),
                "venue_a": venue_a,
                "venue_b": venue_b,
                "lag_buckets": lag,
                "correlation": correlation,
                "observations": int(venue_returns.iloc[:, 1].shift(lag).notna().sum()),
            }
        )
leadership = pd.DataFrame(
    leadership_rows,
    columns=[
        "instrument_id",
        "venue_a",
        "venue_b",
        "lag_buckets",
        "correlation",
        "observations",
    ],
)
display(spread_summary)
display(leadership)
if not synchronized_spreads.empty:
    synchronized_spreads.pivot(index="event_ts", columns="instrument_id", values="spread_bps").plot(
        figsize=(12, 4), title="Cross-venue VWAP spread", alpha=0.8
    )
    plt.axhline(0, color="black", linewidth=0.7)
    plt.ylabel("basis points")
    plt.show()

For leadership, each row estimates `corr(return_A[t], return_B[t-lag])`. A positive lag means B's earlier return is compared with A now; the strongest correlation is only a lead/lag hypothesis. Sparse buckets, common information, venue clocks, and sampling can create the same pattern. Spreads are **not executable arbitrage** because fees, executable depth, transfer delay, and order latency are absent.

## 9. Data-product evolution

Translate named measurements into layer-specific products, validation rules, evidence strength, and priorities.

In [ ]:
event_seconds = event_duration.total_seconds()
if len(raw_df) >= 20_000 and event_seconds >= 300:
    evidence_strength = "strong within this capture"
elif len(raw_df) >= 2_000 and event_seconds >= 60:
    evidence_strength = "directional"
else:
    evidence_strength = "weak: expand the capture"

recommendation_rows = []


def recommend(
    *,
    metric: str,
    measured_value: object,
    proposed_product: str,
    layer: str,
    validation_or_sla: str,
    priority: str,
    strength: str = evidence_strength,
) -> None:
    if not metric or measured_value is None:
        raise ValueError("Every recommendation needs a named metric and measured value.")
    recommendation_rows.append(
        {
            "metric": metric,
            "measured_value": measured_value,
            "evidence_strength": strength,
            "proposed_product": proposed_product,
            "layer": layer,
            "validation_or_sla": validation_or_sla,
            "priority": priority,
        }
    )


backfill_share = float(raw_df["is_backfill"].astype(bool).mean())
overall_p95_latency = float(raw_df["latency_ms"].quantile(0.95))
recommend(
    metric="backfill_share",
    measured_value=backfill_share,
    proposed_product="Immutable trade landing with source, backfill, Kafka, and ingestion metadata",
    layer="Bronze",
    validation_or_sla="Preserve source/is_backfill and Kafka coordinates for 100% of rows",
    priority="build now",
)
recommend(
    metric="duplicate_row_rate",
    measured_value=duplicate_rate,
    proposed_product="Natural-key deduplicated trades",
    layer="Silver",
    validation_or_sla="Unique (venue, venue_symbol, trade_id) after processing",
    priority="build now",
)
recommend(
    metric="conflicting_duplicate_keys",
    measured_value=conflicting_keys,
    proposed_product="Conflicting-trade quarantine with raw lineage",
    layer="Silver",
    validation_or_sla="No conflicting natural key enters the trusted table",
    priority="build now" if conflicting_keys else "validate next",
)
recommend(
    metric="latency_p95_ms",
    measured_value=overall_p95_latency,
    proposed_product="Freshness and latency service-level dashboard",
    layer="Observability",
    validation_or_sla=(
        "Track p50/p95/p99 by venue; investigate sustained p95 above "
        f"{overall_p95_latency:,.0f} ms baseline"
    ),
    priority="build now",
)
recommend(
    metric="end_freshness_s",
    measured_value=freshness_s,
    proposed_product="Per-venue stream freshness monitor",
    layer="Observability",
    validation_or_sla="Alert on sustained freshness breach relative to rolling baseline",
    priority="build now",
)
recommend(
    metric="partition_count_cv",
    measured_value=partition_cv,
    proposed_product="Partition throughput and hot-key monitor",
    layer="Observability",
    validation_or_sla="Track count and byte skew; investigate sustained CV above capture baseline",
    priority="validate next",
)
recommend(
    metric="event_time_bar_count",
    measured_value=len(activity),
    proposed_product="Per-venue and consolidated OHLCV/VWAP bars",
    layer="Gold",
    validation_or_sla="VWAP within low/high; publish only after deduplication and watermarking",
    priority="build now",
)
recommend(
    metric="buy_sell_notional_imbalance",
    measured_value=notional_imbalance,
    proposed_product="Instrument activity, trade-size, and directional-imbalance features",
    layer="Gold",
    validation_or_sla="Publish with observation count, window, and venue coverage",
    priority="validate next",
)
if not volatility_ranking.empty:
    recommend(
        metric="ranked_instruments_with_volatility",
        measured_value=len(volatility_ranking),
        proposed_product="Rolling realized-volatility features",
        layer="Gold",
        validation_or_sla=(
            "Suppress output below five return observations and retain window metadata"
        ),
        priority="validate next",
    )
if not spread_summary.empty:
    recommend(
        metric="cross_venue_spread_samples",
        measured_value=int(spread_summary["samples"].sum()),
        proposed_product="Synchronized cross-venue VWAP and spread research table",
        layer="Gold",
        validation_or_sla=(
            "Require two venues, synchronized buckets, sample count, "
            "and explicit non-executable label"
        ),
        priority="validate next",
    )

recommendations = pd.DataFrame(recommendation_rows)
assert not recommendations.empty
assert recommendations["metric"].str.len().gt(0).all()
assert recommendations["measured_value"].notna().all()
priority_order = {"build now": 0, "validate next": 1, "defer": 2}
recommendations = recommendations.sort_values(
    by="priority", key=lambda values: values.map(priority_order)
).reset_index(drop=True)
recommendations

These priorities are hypotheses grounded in the displayed capture, not permanent architecture decisions. Re-run `deep` across representative market regimes before turning directional evidence into contractual thresholds. The notebook remains read-only: it does not publish these results or modify the source stream.